# 00 — Pipeline Configuration

This notebook documents how to configure the pipeline before running notebooks 01–08.
It creates no data files — run it to verify your setup and pre-create the cache directories.

## Two run modes

| Mode | sample_size | device | batch_size | When to use |
|---|---|---|---|---|
| **CPU small** | 1 000 | cpu | 32 | Local dev, fast iteration (~minutes) |
| **CPU medium** | 5 000 | cpu | 64 | Local benchmarking (default, ~30 min for full sweep) |
| **GPU large** | 50 000 | cuda | 512 | University server, full-scale results |

Change `SAMPLE_SIZE` and `DEVICE` in the cell below, then run all downstream notebooks.

In [1]:
import sys
sys.path.insert(0, '..')  # makes notebooks/utils importable

from utils import load_config

# ── Edit these two lines to switch modes ──────────────────────────────────────
SAMPLE_SIZE = 5_000
DEVICE      = "cpu"       # "cpu" or "cuda"
# ─────────────────────────────────────────────────────────────────────────────

cfg = load_config(sample_size=SAMPLE_SIZE, device=DEVICE)
cfg.ensure_dirs()
print(cfg)

PipelineConfig(
  sample_size  = 5,000
  data_file    = sample_restaurants_5k.jsonl
  device       = cpu
  batch_size   = 64
  seed         = 42
  cache_dir    = /home/dominik/Documents/Informatik/4_Semester/NLP_Projekt/ReviewScope/data/cache
)


## Data file check

Verify that the sample files produced by notebooks 01–02 are present.

In [2]:
import json

expected_files = [
    "sample_5k.jsonl",
    "sample_restaurants_5k.jsonl",
    "sample_restaurants_10k.jsonl",
]

for fname in expected_files:
    path = cfg.cache_dir / fname
    if path.exists():
        with open(path) as f:
            n = sum(1 for _ in f)
        print(f"  ✓  {fname:45s}  {n:>6,} rows")
    else:
        print(f"  ✗  {fname:45s}  NOT FOUND — run notebooks 01 / 02 first")

  ✓  sample_5k.jsonl                                 5,000 rows
  ✓  sample_restaurants_5k.jsonl                     5,000 rows
  ✓  sample_restaurants_10k.jsonl                   10,000 rows


## Cache directory check

In [3]:
for subdir in ["embeddings", "umap", "clustering", "bertopic"]:
    path = cfg.cache_dir / subdir
    files = list(path.glob("*")) if path.exists() else []
    print(f"  {subdir:12s}  {len(files):>3} file(s)")

  embeddings      0 file(s)
  umap            0 file(s)
  clustering      0 file(s)
  bertopic        0 file(s)


## Results log check

In [4]:
from utils import load_results

df = load_results(cfg.results_csv)
print(f"Total logged experiments: {len(df)}")
if len(df) > 0:
    print(df[["pipeline", "embedding_model", "clustering_algo", "silhouette", "runtime_s"]]
          .to_string(index=False))

Total logged experiments: 0


## Model size reference

### CPU tier — 16 GB RAM, no CUDA

| Model | Params | Size | Instruction? |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 22 M | ~90 MB | no |
| `all-mpnet-base-v2` | 110 M | ~440 MB | no |
| `hkunlp/instructor-base` | 110 M | ~440 MB | yes |
| `hkunlp/instructor-large` | 335 M | ~1.3 GB | yes |
| `intfloat/multilingual-e5-large-instruct` | 560 M | ~2.2 GB | yes |
| `Qwen/Qwen3-Embedding-0.6B` | 600 M | ~2.4 GB fp32 | yes |
| `BAAI/bge-m3` | 570 M | ~2.3 GB | no |

### GPU tier — university server

| Model | Params | VRAM |
|---|---|---|
| `hkunlp/instructor-xl` | 1.5 B | ~6 GB |
| `Qwen/Qwen3-Embedding-4B` | 4 B | ~8 GB |
| `intfloat/e5-mistral-7b-instruct` | 7 B | ~14 GB |

### Instruction slugs used in cache filenames

| Slug | Instruction text |
|---|---|
| `no_inst` | (none — direct encode) |
| `generic` | `"Represent the restaurant review for topic clustering:"` |
| `domain` | `"Represent the restaurant review for clustering by theme (food quality, service, ambiance, price):"` |
| `sentiment` | `"Represent the restaurant review to capture the main sentiment and opinion:"` |